In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.coordinates import Longitude, Latitude, Angle
from astropy.io import fits
from astroquery.simbad import Simbad
from astroquery.hips2fits import hips2fits

exoplanets = pd.read_csv('exoplanets.csv')
exoplanets.head(10)

# COLUMN pl_name:         Planet Name
# COLUMN hostname:        Host Name
# COLUMN default_flag:    Default Parameter Set
# COLUMN sy_snum:         Number of Stars
# COLUMN sy_pnum:         Number of Planets
# COLUMN discoverymethod: Discovery Method
# COLUMN disc_year:       Discovery Year
# COLUMN disc_facility:   Discovery Facility
# COLUMN soltype:         Solution Type
# COLUMN pl_controv_flag: Controversial Flag
# COLUMN pl_refname:      Planetary Parameter Reference
# COLUMN pl_orbper:       Orbital Period [days]
# COLUMN pl_orbsmax:      Orbit Semi-Major Axis [au]
# COLUMN pl_rade:         Planet Radius [Earth Radius]
# COLUMN pl_radj:         Planet Radius [Jupiter Radius]
# COLUMN pl_bmasse:       Planet Mass or Mass*sin(i) [Earth Mass]
# COLUMN pl_bmassj:       Planet Mass or Mass*sin(i) [Jupiter Mass]
# COLUMN pl_bmassprov:    Planet Mass or Mass*sin(i) Provenance
# COLUMN pl_orbeccen:     Eccentricity
# COLUMN pl_insol:        Insolation Flux [Earth Flux]
# COLUMN pl_eqt:          Equilibrium Temperature [K]
# COLUMN ttv_flag:        Data show Transit Timing Variations
# COLUMN st_refname:      Stellar Parameter Reference
# COLUMN st_spectype:    Spectral Type
# COLUMN st_teff:        Stellar Effective Temperature [K]
# COLUMN st_rad:         Stellar Radius [Solar Radius]
# COLUMN st_mass:        Stellar Mass [Solar mass]
# COLUMN st_met:         Stellar Metallicity [dex]
# COLUMN st_metratio:    Stellar Metallicity Ratio
# COLUMN st_logg:        Stellar Surface Gravity [log10(cm/s**2)]
# COLUMN sy_refname:     System Parameter Reference
# COLUMN rastr:          RA [sexagesimal]
# COLUMN ra:             RA [deg]
# COLUMN decstr:         Dec [sexagesimal]
# COLUMN dec:            Dec [deg]
# COLUMN sy_dist:        Distance [pc]
# COLUMN sy_vmag:        V (Johnson) Magnitude
# COLUMN sy_kmag:        Ks (2MASS) Magnitude
# COLUMN sy_gaiamag:     Gaia Magnitude
#COLUMN  rowupdate:      Date of Last Update
#COLUMN  pl_pubdate:     Planetary Parameter Reference Publication Date
#COLUMN  releasedate:    Release Date

,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,ra,decstr,dec,sy_dist,sy_vmag,sy_kmag,sy_gaiamag,rowupdate,pl_pubdate,releasedate
0,11 Com b,11 Com,0,2,1,Radial Velocity,2007,Xinglong Station,Published Confirmed,0,...,185.178779,+17d47m35.71s,17.793252,93.1846,4.72307,2.282,4.44038,2014-07-23,2011-08,2014-07-23
1,11 Com b,11 Com,1,2,1,Radial Velocity,2007,Xinglong Station,Published Confirmed,0,...,185.178779,+17d47m35.71s,17.793252,93.1846,4.72307,2.282,4.44038,2023-09-19,2023-12,2023-09-19
2,11 Com b,11 Com,0,2,1,Radial Velocity,2007,Xinglong Station,Published Confirmed,0,...,185.178779,+17d47m35.71s,17.793252,93.1846,4.72307,2.282,4.44038,2014-05-14,2008-01,2014-05-14
3,11 UMi b,11 UMi,1,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,229.274595,+71d49m26.19s,71.823943,125.3210,5.01300,1.939,4.56216,2018-09-04,2017-03,2018-09-06
4,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,229.274595,+71d49m26.19s,71.823943,125.3210,5.01300,1.939,4.56216,2018-04-25,2011-08,2014-07-23
5,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,229.274595,+71d49m26.19s,71.823943,125.3210,5.01300,1.939,4.56216,2018-04-25,2009-10,2014-05-14
6,14 And b,14 And,1,1,1,Radial Velocity,2008,Okayama Astrophysical Observatory,Published Confirmed,0,...,352.824150,+39d14m09.01s,39.235837,75.4392,5.23133,2.331,4.91781,2023-09-19,2023-12,2023-09-19
7,14 And b,14 And,0,1,1,Radial Velocity,2008,Okayama Astrophysical Observatory,Published Confirmed,0,...,352.824150,+39d14m09.01s,39.235837,75.4392,5.23133,2.331,4.91781,2014-05-14,2008-12,2014-05-14
8,14 And b,14 And,0,1,1,Radial Velocity,2008,Okayama Astrophysical Observatory,Published Confirmed,0,...,352.824150,+39d14m09.01s,39.235837,75.4392,5.23133,2.331,4.91781,2014-07-23,2011-08,2014-07-23
9,14 Her b,14 Her,0,1,2,Radial Velocity,2002,W. M. Keck Observatory,Published Confirmed,0,...,242.602101,+43d48m58.90s,43.816362,17.9323,6.61935,4.714,6.38300,2018-04-25,2003-01,2014-08-21


In [4]:
exoplanets_forstudy = exoplanets[['pl_name', 'soltype', 'dec', 'sy_dist', 'pl_orbper', 'pl_orbsmax', 'st_spectype', 'st_teff', 'st_logg', 'sy_gaiamag']]
exoplanets_forstudy.head()
seen = []
l, _ = exoplanets_forstudy.shape
while(l > 0):
    name = exoplanets_forstudy.loc[l-1, 'pl_name']
    if(name in seen):
        exoplanets_forstudy.drop(l-1, inplace=True)
    else:
        seen.append(name)
    l -= 1

In [ ]:
exoplanets_forstudy.head()

,pl_name,soltype,dec,sy_dist,pl_orbper,pl_orbsmax,st_spectype,st_teff,st_logg,sy_gaiamag
2,11 Com b,Published Confirmed,17.793252,93.1846,326.03,1.29,G8 III,4742.0,2.31,4.44038
5,11 UMi b,Published Confirmed,71.823943,125.3210,516.22,1.54,K4 III,4340.0,1.60,4.56216
8,14 And b,Published Confirmed,39.235837,75.4392,NaN,0.68,NaN,NaN,NaN,4.91781
17,14 Her b,Published Confirmed,43.816362,17.9323,1773.40,2.77,K0 V,NaN,NaN,6.38300
23,16 Cyg B b,Published Confirmed,50.516824,21.1397,800.80,1.60,G2.5 V,5760.0,4.35,6.06428


: 